# Chapter 2 - Circles
*Analytic Geometry and Calculus with Python* (4th edition), S. E. Moore.

Run each cell with **Shift + Enter**. All the libraries used (NumPy, SymPy, Matplotlib, SciPy) are pre-installed on Google Colab.

## Listing 2.1: Centre and radius from the general equation of a circle

In [ ]:
# Listing 2.1: Centre and radius from the general equation of a circle
import sympy as sp

def centre_radius(g2, f2, c):
    """For x^2 + y^2 + (2g)x + (2f)y + c = 0, given 2g, 2f and c."""
    g, f = sp.Rational(g2) / 2, sp.Rational(f2) / 2
    r2 = g**2 + f**2 - c
    if r2 < 0:
        return (-g, -f), None          # no real circle
    return (-g, -f), sp.sqrt(r2)

# Checking the worked example: x^2 + y^2 + 3x - 6y - 1 = 0
print(centre_radius(3, -6, -1))

# Exercise-style checks
for coeffs in [(6, -10, -9), (0, 4, 0), (2, 2, 10)]:
    centre, r = centre_radius(*coeffs)
    print(f"2g={coeffs[0]:>3}, 2f={coeffs[1]:>3}, c={coeffs[2]:>3}  ->  centre {centre}, radius {r}")

# SymPy can complete the square for us too
x, y = sp.symbols("x y")
expr = x**2 + y**2 + 3*x - 6*y - 1
print(sp.expand((x + sp.Rational(3, 2))**2 + (y - 3)**2 - sp.Rational(49, 4)) == expr)


## Listing 2.2: Drawing circles with the parametric form x = a + r cos t, y = b + r sin t

In [ ]:
# Listing 2.2: Drawing circles with the parametric form x = a + r cos t, y = b + r sin t
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 2*np.pi, 300)

plt.figure(figsize=(5, 5))
# Lake Bosomtwe is roughly circular, about 8 km across (radius ~ 4 km).
a, b, r = 0, 0, 4
plt.fill(a + r*np.cos(t), b + r*np.sin(t), color="tab:blue", alpha=0.25,
         label="Lake Bosomtwe (approx.), r = 4 km")

# Circle of Example 2.1: centre (-3/2, 3), radius 7/2
a, b, r = -1.5, 3, 3.5
plt.plot(a + r*np.cos(t), b + r*np.sin(t), "r", label="x²+y²+3x−6y−1=0")
plt.plot(a, b, "r+")

plt.gca().set_aspect("equal")
plt.grid(alpha=0.3)
plt.legend(loc="lower right", fontsize=8)
plt.show()

print(f"Area of a circular lake of radius 4 km: {np.pi*4**2:.1f} km^2")


## Listing 2.3: The circle on a given diameter - two methods

In [ ]:
# Listing 2.3: The circle on a given diameter - two methods
import sympy as sp

x, y = sp.symbols("x y")
A, B = (10, 2), (3, 8)

# Method 1: angle in a semicircle is 90 degrees -> (x-x1)(x-x2) + (y-y1)(y-y2) = 0
eq1 = sp.expand((x - A[0])*(x - B[0]) + (y - A[1])*(y - B[1]))
print("Method 1:", eq1, "= 0")

# Method 2: centre = midpoint, radius = half the diameter
C = sp.Point(A).midpoint(sp.Point(B))
r2 = C.distance(sp.Point(B))**2
eq2 = sp.expand((x - C.x)**2 + (y - C.y)**2 - r2)
print("Method 2:", eq2, "= 0   centre", C, " radius^2 =", r2)
print("Same circle?", sp.simplify(eq1 - eq2) == 0)


## Listing 2.4: The circle through three points - a linear system for g, f, c

In [ ]:
# Listing 2.4: The circle through three points - a linear system for g, f, c
import numpy as np
import sympy as sp

def circle_through(P1, P2, P3):
    """Solve x^2+y^2+2gx+2fy+c=0 through three points. Returns (g, f, c)."""
    M, rhs = [], []
    for (x, y) in (P1, P2, P3):
        M.append([2*x, 2*y, 1])          # unknowns g, f, c
        rhs.append(-(x**2 + y**2))
    return np.linalg.solve(np.array(M, float), np.array(rhs, float))

# Checking the worked example: (2,1), (0,2), (1,0)
g, f, c = circle_through((2, 1), (0, 2), (1, 0))
print(f"g = {g:.4f}, f = {f:.4f}, c = {c:.4f}")

# The same, exactly, with SymPy
G, F, Cc, x, y = sp.symbols("g f c x y")
circle = x**2 + y**2 + 2*G*x + 2*F*y + Cc
sol = sp.solve([circle.subs({x: px, y: py}) for px, py in [(2, 1), (0, 2), (1, 0)]], [G, F, Cc])
print("Exact:", sol, " ->", sp.expand(3*circle.subs(sol)), "= 0")

# A telecom mast equidistant from three towns A(0,0), B(10,2), C(4,8)  (km)
g, f, c = circle_through((0, 0), (10, 2), (4, 8))
centre = (-g, -f)
radius = np.sqrt(g**2 + f**2 - c)
print(f"Mast at ({centre[0]:.3f}, {centre[1]:.3f}); each town is {radius:.3f} km away")


## Listing 2.5: Tangent and normal to a circle at a point

In [ ]:
# Listing 2.5: Tangent and normal to a circle at a point
import sympy as sp

x, y = sp.symbols("x y")
S = x**2 + y**2 - 4*x + 2*y            # circle x^2 + y^2 - 4x + 2y = 0
P = (0, 0)
print("P on circle?", S.subs({x: P[0], y: P[1]}) == 0)

# The normal at P passes through the centre (-g, -f) = (2, -1)
centre = (2, -1)
m_radius = sp.Rational(centre[1] - P[1], centre[0] - P[0])
m_tangent = -1 / m_radius
tangent = sp.expand(y - P[1] - m_tangent*(x - P[0]))
normal = sp.expand(y - P[1] - m_radius*(x - P[0]))
print("gradient of radius =", m_radius, ",  gradient of tangent =", m_tangent)
print("tangent:", tangent, "= 0")
print("normal :", normal, "= 0")

# The 'replace one x by x1' rule: x x1 + y y1 + g(x + x1) + f(y + y1) + c = 0
g, f, c = -2, 1, 0
rule = sp.expand(x*P[0] + y*P[1] + g*(x + P[0]) + f*(y + P[1]) + c)
print("by the rule:", rule, "= 0")


## Listing 2.6: Where does a road cross a mast's coverage circle?

In [ ]:
# Listing 2.6: Where does a road cross a mast's coverage circle?
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

x = sp.symbols("x")
R = 5                                   # coverage radius (km), mast at the origin

def crossing(m, c):
    """Intersect y = m x + c with x^2 + y^2 = R^2; classify with the discriminant."""
    quad = sp.expand(x**2 + (m*x + c)**2 - R**2)
    A, B, C = [quad.coeff(x, k) for k in (2, 1, 0)]
    disc = B**2 - 4*A*C
    xs = sp.solve(quad, x)
    pts = [(float(xi), float(m*xi + c)) for xi in xs if xi.is_real]
    kind = "2 points" if disc > 0 else ("tangent" if disc == 0 else "misses")
    return kind, disc, pts

roads = {"Road 1: y = x/2 + 1": (sp.Rational(1, 2), 1),
         "Road 2: y = 5":       (0, 5),
         "Road 3: y = -x + 9":  (-1, 9)}
plt.figure(figsize=(5, 5))
t = np.linspace(0, 2*np.pi, 300)
plt.fill(R*np.cos(t), R*np.sin(t), alpha=0.15, label="coverage")
xx = np.linspace(-7, 9, 50)
for name, (m, c) in roads.items():
    kind, disc, pts = crossing(m, c)
    print(f"{name:<20} discriminant = {float(disc):8.2f}  -> {kind}", [tuple(round(v, 3) for v in p) for p in pts])
    if len(pts) == 2:
        L = np.hypot(pts[0][0] - pts[1][0], pts[0][1] - pts[1][1])
        print(f"{'':20} length of road inside coverage = {L:.3f} km")
    plt.plot(xx, float(m)*xx + c, label=name)
plt.plot(0, 0, "k^", ms=10)
plt.xlim(-7, 9); plt.ylim(-7, 9)
plt.gca().set_aspect("equal"); plt.grid(alpha=0.3); plt.legend(fontsize=7, loc="lower left")
plt.show()

# Exam question (2021): x - 2y - 1 = 0 and x^2 + y^2 + 4x - 2y - 5 = 0
X, Y = sp.symbols("X Y")
print(sp.solve([X - 2*Y - 1, X**2 + Y**2 + 4*X - 2*Y - 5], [X, Y]))


## Listing 2.7: Two coverage circles - how do they meet, and where is the common chord?

In [ ]:
# Listing 2.7: Two coverage circles - how do they meet, and where is the common chord?
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

def relative_position(c1, r1, c2, r2):
    d = math.dist(c1, c2)
    if d > r1 + r2:            return d, "separate (no common point)"
    if d == r1 + r2:           return d, "touch externally"
    if d == abs(r1 - r2):      return d, "touch internally"
    if d < abs(r1 - r2):       return d, "one inside the other"
    return d, "intersect at two points"

print(relative_position((0, 0), 5, (6, 2), 4))
print(relative_position((0, 0), 5, (8, 6), 5))
print(relative_position((0, 0), 5, (1, 1), 2))

# Common chord: subtract the two equations
x, y = sp.symbols("x y")
S1 = x**2 + y**2 - 25                       # mast A at (0,0), radius 5 km
S2 = (x - 6)**2 + (y - 2)**2 - 16           # mast B at (6,2), radius 4 km
chord = sp.expand(S1 - S2)
print("Common chord:", chord, "= 0")
pts = sp.solve([S1, S2], [x, y])
pts = [(float(a), float(b)) for a, b in pts]
print("Intersection points:", [tuple(round(v, 4) for v in p) for p in pts])

t = np.linspace(0, 2*np.pi, 300)
plt.figure(figsize=(5.5, 4.5))
plt.fill(5*np.cos(t), 5*np.sin(t), alpha=0.2, label="mast A")
plt.fill(6 + 4*np.cos(t), 2 + 4*np.sin(t), alpha=0.2, label="mast B")
plt.plot(*zip(*pts), "k-o", label="common chord")
plt.gca().set_aspect("equal"); plt.grid(alpha=0.3); plt.legend(fontsize=8)
plt.show()


## Listing 2.8: Locating a phone from its distances to three masts (trilateration)

In [ ]:
# Listing 2.8: Locating a phone from its distances to three masts (trilateration)
import numpy as np

masts = np.array([[0.0, 0.0], [10.0, 0.0], [4.0, 9.0]])   # km
d = np.array([5.000, 8.062, 5.099])                       # measured distances, km

# Circle i:  x^2 + y^2 - 2 a_i x - 2 b_i y + (a_i^2 + b_i^2 - d_i^2) = 0
# Subtracting circle 1 from circles 2 and 3 removes x^2 + y^2 and leaves two LINES
# (the common chords), whose intersection is the phone.
(a1, b1), (a2, b2), (a3, b3) = masts
M = np.array([[2*(a2 - a1), 2*(b2 - b1)],
              [2*(a3 - a1), 2*(b3 - b1)]])
rhs = np.array([d[0]**2 - d[1]**2 + a2**2 - a1**2 + b2**2 - b1**2,
                d[0]**2 - d[2]**2 + a3**2 - a1**2 + b3**2 - b1**2])
phone = np.linalg.solve(M, rhs)
print("Estimated position of the phone:", np.round(phone, 3))
print("Check distances:", np.round(np.linalg.norm(masts - phone, axis=1), 3))


## Listing 2.9: From the Rhind papyrus to Archimedes - approximating pi

In [ ]:
# Listing 2.9: From the Rhind papyrus to Archimedes - approximating pi
import math

print(f"Rhind papyrus (c. 1550 BCE): (16/9)^2 = {(16/9)**2:.6f}")
print(f"Archimedes' bounds: 223/71 = {223/71:.6f} < pi < 22/7 = {22/7:.6f}")

# Archimedes: perimeters of regular polygons inside and outside a circle of diameter 1.
# Start with hexagons and keep doubling the number of sides.
n = 6
inner = 3.0                       # inscribed hexagon: perimeter 6 * (1/2)
outer = 2 * math.sqrt(3)          # circumscribed hexagon
print(f"{'sides':>6} {'inscribed':>12} {'circumscribed':>14}")
for _ in range(5):
    print(f"{n:>6} {inner:>12.6f} {outer:>14.6f}")
    outer = 2 * inner * outer / (inner + outer)   # harmonic mean
    inner = math.sqrt(inner * outer)              # geometric mean
    n *= 2
print(f"{n:>6} {inner:>12.6f} {outer:>14.6f}")
print(f"{'pi':>6} {math.pi:>12.6f}")
